### "SubscribeMax" — a subscription box company that sells two premium product lines: NCLT → "NutriBox" (nutrition subscription) and VNCLT → "VitaBox" (vitamin subscription). The fraud/analytics team monitors customers who have suspiciously high coverage-value ratios and multiple subscriptions under the same owner — exactly mirroring the original insurance fraud detection logic.

|     Table Name             | Represents                                                   |
| -------------------------- | ------------------------------------------------------------ |
| t_subscription_master      | Subscription master record                                   |
| t_subscription_customer    | Customer linked to subscription                              |
| t_subscription_plan_detail | Plan/coverage detail per subscription                        |
| t_product_catalog          | Product catalog (NutriBox/VitaBox)                           |
| t_group_contract           | Group contract info                                          |
| t_cv_ratio_category        | CV ratio category lookup                                     |
| t_payment_history          | Payment transaction history                                  |
| t_plan_key_master          | Plan key / payment term master                               |
| t_premium_paid_detail      | Total premium paid detail                                    |
| t_customer_kana            | Customer phonetic name (kept as kana concept → display name) |
| t_agreement_agent_snapshot | Agent linked to agreement                                    |
| t_agent_hierarchy_snapshot | Agent hierarchy master                                       |
| t_agent_master_snapshot    | Agent master                                                 |
| t_agreement_agent_history  | Agreement-agent history                                      |

|     Column                              | Meaning                            |
| --------------------------------------- | ---------------------------------- |
| SUB_NO                                  | Subscription number                |
| PLAN_UNIT                               | Plan unit identifier               |
| OWNER_ID                                | Subscription owner ID              |
| ACTIVATION_DT                           | Original activation date           |
| CHANGE_EFF_DT                           | Change effective date              |
| COVERAGE_START_DT                       | Coverage/responsibility start date |
| CONTRACT_DT                             | Contract start date                |
| ORIG_CONTRACT_DT                        | Original contract date             |
| STATUS_CD                               | Subscription status code           |
| TERMINATION_CD                          | Termination reason code            |
| BASE_PREMIUM                            | Base monthly premium amount        |
| PAY_FREQUENCY_CD                        | Payment frequency code             |
| PLAN_EFF_YM                             | Plan effective year-month          |
| PLAN_START_DT                           | Plan responsibility start date     |
| PLAN_PHASE                              | Plan phase                         |
| PRODUCT_CD                              | Product base code                  |
| PRODUCT_SUB_CD                          | Product sub code                   |
| PRODUCT_LINE                            | Product line (NutriBox/VitaBox)    |
| PLAN_NAME                               | Plan name                          |
| OWNER_LAST_NM                           | Owner last name                    |
| OWNER_FIRST_NM                          | Owner first name                   |
| OWNER_DISPLAY_NM                        | Owner display name                 |
| OWNER_PHONE                             | Owner phone                        |
| OWNER_ADDR1/2/3                         | Owner address                      |
| OWNER_TYPE                              | Owner type                         |
| INSURED_LAST_NM                         | Insured/subscriber last name       |
| INSURED_FIRST_NM                        | Insured/subscriber first name      |
| INSURED_DISPLAY_NM                      | Insured display name               |
| INSURED_DOB                             | Insured date of birth              |
| INSURED_GENDER                          | Insured gender                     |
| GROUP_CONTRACT_NO                       | Group contract number              |
| RATIO_TYPE                              | Ratio type (Group/Individual)      |
| CV_TIER                                 | CV ratio tier                      |
| PLAN_MATURITY_AGE                       | Plan maturity age                  |
| PAYMENT_TERM_YR                         | Payment term in years              |
| PAYMENT_YM                              | Payment year-month                 |
| PAYMENT_MODE                            | Payment mode                       |
| PREMIUM_AMT                             | Premium amount                     |
| NET_PREMIUM_PAID                        | Net premium paid to date           |
| AGENT_L9_CD                             | Agent Level 9 code                 |
| AGENT_L9_NM                             | Agent Level 9 name                 |
| AGENT_L3_CD / AGENT_L1_CD / AGENT_L4_CD | Agent hierarchy codes              |
| AGENT_CHANNEL                           | Agent channel/line                 |
| English labels                          | See CASE block below               |

In [ ]:
-- ============================================================
-- STEP 0: Enable Adaptive Query Execution
-- ============================================================
SET spark.sql.adaptive.enabled = true;
SET spark.sql.adaptive.coalescePartitions.enabled = true;
SET spark.sql.sources.bucketing.enabled = true;
SET spark.sql.sources.bucketing.autoBucketedScanEnabled = true;

-- ============================================================
-- STEP 1: Materialize core pool ONCE — cached in memory
-- Replaces 3 separate re-scans of the same 5 base tables
-- ============================================================
CREATE OR REPLACE TEMP TABLE SUBSCRIPTION_POOL_CACHED
USING DELTA
PARTITIONED BY (PRODUCT_LINE)
CLUSTERED BY (SUB_NO) INTO 64 BUCKETS
AS
SELECT
    SUB.SUB_NO,
    PD.PLAN_UNIT,
    CUST.OWNER_ID,
    SUB.ACTIVATION_DT,
    PD.PRODUCT_CD,
    PD.PRODUCT_SUB_CD,
    PD.PLAN_EFF_YM,
    PD.PLAN_PHASE,
    SUB.PAY_FREQUENCY_CD,
    SUB.BASE_PREMIUM,
    SUB.CONTRACT_DT,
    SUB.GROUP_CONTRACT_NO,
    CUST.INSURED_GENDER,
    CUST.INSURED_DOB,
    CAT.PRODUCT_LINE,
    CAT.PLAN_NAME,
    GRP.GROUP_CONTRACT_NO  AS GRP_CONTRACT_NO,
    GRP.PAY_FREQUENCY_CD   AS GRP_PAY_FREQ_CD,
    CVR.CV_TIER
FROM subscribemax.t_subscription_master SUB
INNER JOIN subscribemax.t_subscription_customer CUST
    ON SUB.SUB_NO = CUST.SUB_NO
    AND CUST.OWNER_TYPE = 'C'
INNER JOIN subscribemax.t_subscription_plan_detail PD
    ON SUB.SUB_NO = PD.SUB_NO
    AND PD.PLAN_PHASE = '1'
    AND SUB.BASE_PREMIUM * (12 / SUB.PAY_FREQUENCY_CD) < 300000
INNER JOIN subscribemax.t_product_catalog CAT
    ON CAT.PRODUCT_CD     = PD.PRODUCT_CD
    AND CAT.PRODUCT_SUB_CD = PD.PRODUCT_SUB_CD
    AND CAT.PRODUCT_LINE   IN ('NutriBox', 'VitaBox')
LEFT JOIN subscribemax.t_group_contract GRP
    ON GRP.GROUP_CONTRACT_NO = SUB.GROUP_CONTRACT_NO
LEFT JOIN analytics.t_cv_ratio_category CVR
    ON PD.PRODUCT_CD       = CVR.PRODUCT_CD
    AND PD.PRODUCT_SUB_CD  = CVR.PRODUCT_SUB_CD
    AND SUB.PAY_FREQUENCY_CD = CVR.PAY_FREQUENCY_CD
    AND CUST.INSURED_GENDER  = CVR.GENDER
    AND FLOOR(MONTHS_BETWEEN(SUB.CONTRACT_DT, CUST.INSURED_DOB) / 12) = CVR.AGE
    AND CVR.RATIO_TYPE = CASE
        WHEN (GRP.GROUP_CONTRACT_NO IS NOT NULL AND GRP.PAY_FREQ_CD = '01') THEN 'G'
        ELSE 'I'
    END;

CACHE TABLE SUBSCRIPTION_POOL_CACHED;


-- ============================================================
-- MAIN QUERY
-- ============================================================
WITH

-- ----------------------------------------------------------
-- Active subscription pool filtered to fraud-risk criteria
-- Reads from cached table — no base table re-scan
-- ----------------------------------------------------------
ACTIVE_POOL AS (
    SELECT
        SUB_NO,
        PLAN_UNIT,
        OWNER_ID,
        ACTIVATION_DT
    FROM SUBSCRIPTION_POOL_CACHED
    WHERE
        (PRODUCT_LINE = 'VitaBox'  AND CV_TIER = '1')
        OR (PRODUCT_LINE = 'NutriBox' AND INSTR(PLAN_NAME, 'LOW_CV_RIDER') > 0 AND PLAN_EFF_YM >= '202304')
),

-- ----------------------------------------------------------
-- First payment record per subscription
-- Replaced IN (SELECT 4-table-join) with JOIN on cached pool
-- Only RANKA=1 exits this CTE — one row per SUB_NO
-- ----------------------------------------------------------
FIRST_PAYMENT AS (
    SELECT
        PAYMENT_YM,
        SUB_NO,
        PAYMENT_MODE,
        BASE_PREMIUM,
        PREMIUM_AMT,
        ROW_NUMBER() OVER (PARTITION BY SUB_NO ORDER BY PAYMENT_YM ASC)  AS RANKA,
        ROW_NUMBER() OVER (PARTITION BY SUB_NO ORDER BY PAYMENT_YM DESC) AS RANKD
    FROM subscribemax.t_payment_history
    WHERE SUB_NO IN (SELECT SUB_NO FROM SUBSCRIPTION_POOL_CACHED)
),

-- ----------------------------------------------------------
-- Aggregate pool by owner — detects multi-subscription owners
-- ----------------------------------------------------------
POOL_AGGREGATES AS (
    SELECT
        P1.OWNER_ID,
        P1.PLAN_UNIT,
        MIN(P2.ACTIVATION_DT)                                                   AS MIN_ACTIVATION_DT_ALL,
        SUM(CASE WHEN P2.PLAN_UNIT <> P1.PLAN_UNIT THEN 1 ELSE 0 END)           AS CNT_DIFF_PLAN_UNIT,
        MAX(CASE WHEN P2.PLAN_UNIT <> P1.PLAN_UNIT THEN P2.ACTIVATION_DT END)   AS MAX_ACTIVATION_DT_DIFF
    FROM ACTIVE_POOL P1
    INNER JOIN ACTIVE_POOL P2
        ON P1.OWNER_ID = P2.OWNER_ID
    GROUP BY P1.OWNER_ID, P1.PLAN_UNIT
),

-- ----------------------------------------------------------
-- VitaBox maturity age pool — derived from cached table
-- Replaced 5-table re-join with read from cached pool
-- ----------------------------------------------------------
VITABOX_MATURITY_POOL AS (
    SELECT
        IC.SUB_NO,
        IC.PLAN_UNIT,
        IC.OWNER_ID,
        CVR.PLAN_MATURITY_AGE
    FROM SUBSCRIPTION_POOL_CACHED IC
    INNER JOIN analytics.t_cv_ratio_category CVR
        ON IC.PRODUCT_CD      = CVR.PRODUCT_CD
        AND IC.PRODUCT_SUB_CD  = CVR.PRODUCT_SUB_CD
        AND IC.PAY_FREQUENCY_CD = CVR.PAY_FREQUENCY_CD
        AND IC.INSURED_GENDER   = CVR.GENDER
        AND FLOOR(MONTHS_BETWEEN(IC.CONTRACT_DT, IC.INSURED_DOB) / 12) = CVR.AGE
        AND CVR.RATIO_TYPE = CASE
            WHEN (IC.GRP_CONTRACT_NO IS NOT NULL AND IC.GRP_PAY_FREQ_CD = '01') THEN 'G'
            ELSE 'I'
        END
    WHERE IC.PRODUCT_LINE = 'VitaBox'
),

-- ----------------------------------------------------------
-- Pre-aggregated maturity flag per owner
-- Replaces correlated EXISTS — one row per OWNER_ID
-- Downstream uses LEFT JOIN instead of per-row subquery scan
-- ----------------------------------------------------------
MATURITY_AGE_FLAGS AS (
    SELECT
        OWNER_ID,
        COLLECT_SET(PLAN_MATURITY_AGE) AS ALL_MATURITY_AGES,
        COUNT(DISTINCT PLAN_MATURITY_AGE) AS DISTINCT_MATURITY_CNT
    FROM VITABOX_MATURITY_POOL
    GROUP BY OWNER_ID
),

-- ----------------------------------------------------------
-- Main subscription list — full detail with all flags
-- ----------------------------------------------------------
SUBSCRIPTION_LIST AS (
    SELECT
        date_format(current_date(), 'yyyy/MM/dd')                        AS MONITOR_DATE,
        date_format(CAST(SUB.ACTIVATION_DT  AS DATE), 'yyyy/MM/dd')      AS ACTIVATION_DATE,
        date_format(CAST(SUB.CHANGE_EFF_DT  AS DATE), 'yyyy/MM/dd')      AS CHANGE_EFF_DT,
        date_format(CAST(SUB.COVERAGE_START_DT AS DATE), 'yyyy/MM/dd')   AS COVERAGE_START_DT,
        date_format(CAST(SUB.CONTRACT_DT    AS DATE), 'yyyy/MM/dd')      AS CONTRACT_DT,
        SUB.SUB_NO,
        SUB.STATUS_CD,
        SUB.TERMINATION_CD,
        SUB.BASE_PREMIUM,
        SUB.PAY_FREQUENCY_CD,
        SUB.CONTRACT_DT                                                   AS CONTRACT_DATE_RAW,
        PD.PLAN_EFF_YM,
        date_format(CAST(PD.PLAN_START_DT AS DATE), 'yyyy/MM/dd')        AS PLAN_START_DT,
        PD.PLAN_UNIT,
        CAT.PRODUCT_LINE,
        CAT.PLAN_NAME,
        CUST.OWNER_LAST_NM,
        CUST.OWNER_FIRST_NM,
        CUST.OWNER_PHONE,
        CUST.OWNER_ADDR1,
        CUST.OWNER_ADDR2,
        CUST.OWNER_ADDR3,
        CUST.OWNER_ID,
        CUST.OWNER_TYPE,
        CUST.INSURED_LAST_NM,
        CUST.INSURED_FIRST_NM,
        CUST.INSURED_DOB,
        CUST.INSURED_GENDER,
        SUB.ACTIVATION_DT,

        -- Flag: annual premium under threshold
        CASE
            WHEN SUB.BASE_PREMIUM * (12 / SUB.PAY_FREQUENCY_CD) < 300000 THEN 'Y'
            ELSE NULL
        END AS BELOW_THRESHOLD_FLG,

        -- Flag: NutriBox low CV rider plan
        CASE
            WHEN INSTR(CAT.PLAN_NAME, 'LOW_CV_RIDER') > 0 THEN 'Y'
            ELSE NULL
        END AS NUTRIBOX_LOW_CV_FLG,

        -- Flag: VitaBox highest CV tier
        CASE
            WHEN CVR.CV_TIER = 1 THEN 'Y'
            ELSE NULL
        END AS VITABOX_HIGH_CV_FLG,

        -- Group classification based on activation date
        CASE
            WHEN AGG.MIN_ACTIVATION_DT_ALL <= DATE_SUB(CURRENT_DATE(), 2) THEN 'GROUP_A'
            WHEN AGG.MIN_ACTIVATION_DT_ALL =  DATE_SUB(CURRENT_DATE(), 1) THEN 'GROUP_B'
        END AS GROUP_TYPE,

        -- Flag: all subscriptions under owner have same plan unit
        CASE
            WHEN AGG.CNT_DIFF_PLAN_UNIT > 0 THEN 'N'
            ELSE 'Y'
        END AS PLAN_UNIT_MATCH_FLG,

        -- Flag: most recent activation date is yesterday
        CASE
            WHEN AGG.MAX_ACTIVATION_DT_DIFF = DATE_SUB(CURRENT_DATE(), 1) THEN 'Y'
            ELSE 'N'
        END AS EXTRACT_FLG,

        -- Flag: this subscription was activated yesterday
        CASE
            WHEN SUB.ACTIVATION_DT = DATE_SUB(CURRENT_DATE(), 1) THEN 'Y'
            ELSE NULL
        END AS RECENTLY_ACTIVATED_FLG,

        PKM.PAYMENT_TERM_YR,
        MONTHS_BETWEEN(
            TO_DATE(PPD.PAYMENT_PERIOD || '01', 'yyyyMMdd'),
            TO_DATE(PD.PLAN_EFF_YM    || '01', 'yyyyMMdd')
        )                                                                 AS ACTUAL_PAYMENT_MONTHS,
        FP.BASE_PREMIUM * (12 / CAST(FP.PAYMENT_MODE AS INTEGER))        AS INITIAL_ANP,
        FLOOR(MONTHS_BETWEEN(SUB.ORIG_CONTRACT_DT, CLN.INSURED_DOB) / 12) AS INSURED_AGE,

        -- Subscription status in plain English
        CASE
            WHEN sub.status_cd = '01' THEN 'Other'
            WHEN sub.status_cd = '12' THEN 'First Payment Pending'
            WHEN sub.status_cd = '22' THEN 'Active'
            WHEN sub.status_cd = '31' THEN 'Premium Waived'
            WHEN sub.status_cd = '32' THEN 'Premium Waived'
            WHEN sub.status_cd = '33' THEN 'Premium Waived'
            WHEN sub.status_cd = '40' THEN 'First Payment Pending'
            WHEN sub.status_cd = '41' THEN 'Payment Complete'
            WHEN sub.status_cd = '42' THEN 'Single Premium'
            WHEN sub.status_cd = '44' THEN 'Extended'
            WHEN sub.status_cd = '45' THEN 'Paid Up'
            WHEN sub.status_cd = '46' THEN 'Waiver Payment Complete'
            WHEN sub.status_cd = '49' THEN 'Waiver Payment Complete'
            WHEN sub.status_cd = '4A' THEN 'AETI'
            WHEN sub.status_cd = '4B' THEN 'Auto Paid Up'
            WHEN sub.status_cd = '55' THEN 'Lapsed'
            WHEN sub.status_cd = '98' THEN 'Not Yet Activated'
            WHEN sub.status_cd = '99' THEN CASE
                WHEN sub.termination_cd = 'L' THEN 'Death / Critical Illness'
                WHEN sub.termination_cd = 'M' THEN 'Matured'
                WHEN sub.termination_cd = 'N' THEN 'Term Ended'
                WHEN sub.termination_cd = 'P' THEN 'Cancelled'
                WHEN sub.termination_cd = 'Q' THEN 'Lapsed Expired'
                WHEN sub.termination_cd = 'U' THEN 'Terminated'
                WHEN sub.termination_cd = 'V' THEN 'Void'
                WHEN sub.termination_cd = 'W' THEN 'Cooling Off'
                WHEN sub.termination_cd = 'X' THEN 'Fixed Amount Change'
                ELSE 'Closed'
            END
        END AS SUBSCRIPTION_STATUS,

        CDNM.OWNER_DISPLAY_NM,
        CDNM.INSURED_DISPLAY_NM,
        SUB.BASE_PREMIUM * (12 / CAST(SUB.PAY_FREQUENCY_CD AS INTEGER)) AS ANNUAL_PREMIUM,
        PPD.NET_PREMIUM_PAID,
        SUB.ORIG_CONTRACT_DT,
        CVR.PLAN_MATURITY_AGE,

        -- COVERAGE_TERM_FLG: correlated EXISTS replaced with LEFT JOIN on pre-aggregated flags
        CASE
            WHEN MAF.DISTINCT_MATURITY_CNT > 1 THEN 'N'
            ELSE 'Y'
        END AS MATURITY_AGE_MATCH_FLG

    FROM subscribemax.t_subscription_master SUB
    INNER JOIN subscribemax.t_subscription_customer CUST
        ON SUB.SUB_NO = CUST.SUB_NO
        AND CUST.OWNER_TYPE = 'C'
    INNER JOIN subscribemax.t_subscription_plan_detail PD
        ON SUB.SUB_NO = PD.SUB_NO
        AND PD.PLAN_PHASE = '1'
    INNER JOIN subscribemax.t_subscription_customer CLN
        ON CLN.SUB_NO = SUB.SUB_NO
    INNER JOIN subscribemax.t_product_catalog CAT
        ON CAT.PRODUCT_CD      = PD.PRODUCT_CD
        AND CAT.PRODUCT_SUB_CD  = PD.PRODUCT_SUB_CD
        AND CAT.PRODUCT_LINE    IN ('NutriBox', 'VitaBox')
    LEFT JOIN subscribemax.t_group_contract GRP
        ON GRP.GROUP_CONTRACT_NO = SUB.GROUP_CONTRACT_NO
    LEFT JOIN analytics.t_cv_ratio_category CVR
        ON PD.PRODUCT_CD        = CVR.PRODUCT_CD
        AND PD.PRODUCT_SUB_CD   = CVR.PRODUCT_SUB_CD
        AND SUB.PAY_FREQUENCY_CD = CVR.PAY_FREQUENCY_CD
        AND CUST.INSURED_GENDER  = CVR.GENDER
        AND FLOOR(MONTHS_BETWEEN(SUB.CONTRACT_DT, CUST.INSURED_DOB) / 12) = CVR.AGE
        AND CVR.RATIO_TYPE = CASE
            WHEN (GRP.GROUP_CONTRACT_NO IS NOT NULL AND GRP.PAY_FREQ_CD = '01') THEN 'G'
            ELSE 'I'
        END
    LEFT JOIN POOL_AGGREGATES AGG
        ON AGG.OWNER_ID   = CUST.OWNER_ID
        AND AGG.PLAN_UNIT  = PD.PLAN_UNIT
    LEFT JOIN subscribemax.t_plan_key_master PKM
        ON PD.PRODUCT_CD     = PKM.PRODUCT_CD
        AND PD.PRODUCT_SUB_CD = PKM.PRODUCT_SUB_CD
    LEFT JOIN subscribemax.t_premium_paid_detail PPD
        ON SUB.SUB_NO = PPD.SUB_NO
    LEFT JOIN FIRST_PAYMENT FP
        ON FP.SUB_NO = SUB.SUB_NO AND FP.RANKA = 1
    LEFT JOIN subscribemax.t_customer_display_name CDNM
        ON CLN.SUB_NO = CDNM.SUB_NO
    -- Replaces correlated EXISTS — one pre-aggregated row per OWNER_ID
    LEFT JOIN MATURITY_AGE_FLAGS MAF
        ON MAF.OWNER_ID = CUST.OWNER_ID
    WHERE CAT.PRODUCT_LINE IN ('VitaBox', 'NutriBox')
)

-- ============================================================
-- FINAL SELECT
-- ============================================================
SELECT
    SL.MONITOR_DATE,
    SL.ACTIVATION_DATE,
    SL.PLAN_EFF_YM,
    SL.SUB_NO,
    AH.AGENT_L9_CD,
    AH.AGENT_L9_NM,
    AH.AGENT_L3_CD,
    AH.AGENT_L3_NM,
    AH.AGENT_L1_CD,
    AH.AGENT_L1_NM,
    AGNT.MAIN_AGNT_CD_LVL3,
    AGNT.MAIN_AGNT_CD_LVL1,
    AM.CLASS_NM                                                         AS AGENT_L9_CLASS_NM,
    AH.AGENT_CHANNEL,
    AH.AGENT_L9_CLASS_CD,
    AGNT.MAIN_AGNT_CD_LVL9,
    AH.AGENT_L4_CD,
    AH.AGENT_L4_NM,
    AGNT.MAIN_AGNT_CD_LVL4,
    SL.PRODUCT_LINE,
    SL.PLAN_NAME,
    SL.SUBSCRIPTION_STATUS,
    SL.PAYMENT_TERM_YR,
    SL.ACTUAL_PAYMENT_MONTHS,
    SL.INITIAL_ANP,
    SL.CHANGE_EFF_DT,
    SL.PAY_FREQUENCY_CD,
    CASE
        WHEN SL.PAY_FREQUENCY_CD = '01' THEN 'Monthly'
        WHEN SL.PAY_FREQUENCY_CD = '06' THEN 'Semi-Annual'
        WHEN SL.PAY_FREQUENCY_CD = '12' THEN 'Annual'
        WHEN SL.PAY_FREQUENCY_CD = '90' THEN 'Single Pay'
        ELSE NULL
    END                                                                 AS PAYMENT_FREQUENCY_NM,
    SL.BASE_PREMIUM,
    SL.ANNUAL_PREMIUM                                                   AS TOTAL_ANP,
    SL.NET_PREMIUM_PAID,
    SL.ORIG_CONTRACT_DT,
    SL.PLAN_START_DT,
    SL.PLAN_UNIT,
    SL.PLAN_MATURITY_AGE,
    SL.OWNER_TYPE,
    SL.OWNER_ID,
    SL.OWNER_LAST_NM,
    SL.OWNER_FIRST_NM,
    SL.OWNER_DISPLAY_NM,
    ''                                                                  AS OWNER_CLIENT_NO,
    SL.INSURED_LAST_NM,
    SL.INSURED_FIRST_NM,
    SL.INSURED_GENDER,
    SL.INSURED_DISPLAY_NM,
    SL.INSURED_AGE,
    SL.BELOW_THRESHOLD_FLG,
    SL.NUTRIBOX_LOW_CV_FLG,
    SL.VITABOX_HIGH_CV_FLG,
    SL.RECENTLY_ACTIVATED_FLG,
    CASE
        WHEN SL.BELOW_THRESHOLD_FLG = 'Y'
            AND (
                (SL.NUTRIBOX_LOW_CV_FLG = 'Y' AND SL.PLAN_EFF_YM >= '202304')
                OR SL.VITABOX_HIGH_CV_FLG = 'Y'
            )
        THEN SL.GROUP_TYPE
        ELSE NULL
    END                                                                 AS IS_RISK_GROUP,
    CASE
        WHEN SL.BELOW_THRESHOLD_FLG = 'Y'
            AND (
                (SL.NUTRIBOX_LOW_CV_FLG = 'Y' AND SL.PLAN_EFF_YM >= '202304')
                OR SL.VITABOX_HIGH_CV_FLG = 'Y'
            )
        THEN SL.PLAN_UNIT_MATCH_FLG
        ELSE NULL
    END                                                                 AS IS_PLAN_UNIT_MATCH,
    CASE
        WHEN SL.PRODUCT_LINE = 'VitaBox'
        THEN SL.MATURITY_AGE_MATCH_FLG
        ELSE NULL
    END                                                                 AS MATURITY_AGE_MATCH_FLG,
    CASE
        WHEN SL.MATURITY_AGE_MATCH_FLG = 'N'
            OR (
                SL.BELOW_THRESHOLD_FLG = 'Y'
                AND (
                    (SL.NUTRIBOX_LOW_CV_FLG = 'Y' AND SL.PLAN_EFF_YM >= '202304')
                    OR SL.VITABOX_HIGH_CV_FLG = 'Y'
                )
                AND SL.PLAN_UNIT_MATCH_FLG = 'N'
            )
        THEN 'N'
        ELSE 'Y'
    END                                                                 AS FINAL_RISK_FLAG,
    SL.EXTRACT_FLG,
    SL.PLAN_UNIT_MATCH_FLG

FROM SUBSCRIPTION_LIST SL
INNER JOIN analytics.t_agreement_agent_snapshot AA
    ON AA.AGRE_NUM = SL.SUB_NO
INNER JOIN analytics.t_agent_hierarchy_snapshot AH
    ON AA.MAIN_AGNT_CD_LVL1 = AH.AGENT_L1_CD
LEFT JOIN analytics.t_agent_master_snapshot AM
    ON AH.AGENT_L9_CD = AM.AGENT_CD AND AM.AGENT_LEVEL = '9'
LEFT JOIN (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY SRC_SYS_MSTR_CD, AGRE_NUM
            ORDER BY REC_EFF_STRT_TS ASC, SRC_REC_EFF_STRT_TS ASC
        ) AS RNK
    FROM analytics.t_agreement_agent_history
    WHERE SRC_REC_EFF_END_TS = '9999-01-01 00:00:00'
      AND AGRE_NUM IN (SELECT SUB_NO FROM SUBSCRIPTION_LIST)
) AGNT
    ON SL.SUB_NO = AGNT.AGRE_NUM
    AND AGNT.RNK = 1;

-- ============================================================
-- CLEANUP
-- ============================================================
UNCACHE TABLE SUBSCRIPTION_POOL_CACHED;